In [1]:
import sqlite3
import pandas as pd
import networkx as nx
import os

# 1. Estrazione dal database con deduplicazione delle entità (Alias)
db_path = '../data/document_analysis.db'
conn = sqlite3.connect(db_path)

query = """
SELECT 
    COALESCE(ea1.canonical_name, t.actor) AS source,
    COALESCE(ea2.canonical_name, t.target) AS target
FROM rdf_triples t
LEFT JOIN entity_aliases ea1 ON t.actor = ea1.original_name
LEFT JOIN entity_aliases ea2 ON t.target = ea2.original_name
"""
df_raw_edges = pd.read_sql_query(query, conn)
conn.close()

# Pulizia: rimuoviamo valori nulli e auto-referenze (Epstein -> Epstein)
df_clean = df_raw_edges.dropna(subset=['source', 'target']).copy()
df_clean = df_clean[df_clean['source'] != df_clean['target']]

print(f"Totale relazioni grezze estratte: {len(df_clean)}")

Totale relazioni grezze estratte: 106751


In [2]:
# 2. Calcolo dei Pesi e della Distanza
# Raggruppiamo le coppie per contare la frequenza delle interazioni (weight)
edge_weights = df_clean.groupby(['source', 'target']).size().reset_index(name='weight')

# Invertiamo il peso. Se due persone interagiscono tanto (peso alto), la loro 'distanza' 
# deve essere piccolissima per l'algoritmo di Betweenness.
edge_weights['distance'] = 1.0 / edge_weights['weight']

print(f"Totale archi unici (Edges): {len(edge_weights)}")
display(edge_weights.sort_values('weight', ascending=False).head())

Totale archi unici (Edges): 65354


,source,target,weight,distance
28596,Jeffrey Epstein,unknown person A,961,0.001041
59359,unknown person A,Jeffrey Epstein,580,0.001724
26286,Jeffrey Epstein,Reid Weingarten,477,0.002096
24995,Jeffrey Epstein,Donald Trump,467,0.002141
25600,Jeffrey Epstein,Kathy Ruemmler,455,0.002198


In [3]:
# 3. Creazione del Grafo NetworkX
# Usiamo un grafo non orientato (Graph) ideale per studiare la co-occorrenza e i network sociali
G = nx.from_pandas_edgelist(
    edge_weights, 
    source='source', 
    target='target', 
    edge_attr=['weight', 'distance'], # Passiamo entrambi gli attributi!
    create_using=nx.Graph()
)

print(f"Nodi totali nel grafo: {G.number_of_nodes()}")
print(f"Archi totali nel grafo: {G.number_of_edges()}")

Nodi totali nel grafo: 55854
Archi totali nel grafo: 63103


In [4]:
# 4. Calcolo delle Metriche di Centralità
print("Calcolo in corso... Potrebbe richiedere qualche secondo per la Betweenness.")

# A) Degree: Quanto è "popolare" un nodo (somma dei pesi dei suoi legami)
degree_dict = dict(G.degree(weight='weight'))

# B) Betweenness Centrality: Quanto un nodo agisce da "Ponte" tra mondi diversi
# Usiamo 'distance' invece di 'weight' e campioniamo fino a 2000 nodi per alta precisione
k_samples = min(2000, G.number_of_nodes())
betweenness_dict = nx.betweenness_centrality(G, k=k_samples, weight='distance', seed=42)

# C) PageRank: Importanza del nodo basata sull'importanza di chi lo circonda
pagerank_dict = nx.pagerank(G, weight='weight')

# Uniamo tutto in un comodo DataFrame
df_metrics = pd.DataFrame({
    'Node': list(G.nodes()),
    'Degree': [degree_dict[n] for n in G.nodes()],
    'Betweenness_Centrality': [betweenness_dict[n] for n in G.nodes()],
    'PageRank': [pagerank_dict[n] for n in G.nodes()]
})

# Ordiniamo per Betweenness per vedere i veri "Kingmaker" della rete
df_metrics = df_metrics.sort_values(by='Betweenness_Centrality', ascending=False)
print("\nI veri Top 10 Broker della Rete:")
display(df_metrics.head(10))

Calcolo in corso... Potrebbe richiedere qualche secondo per la Betweenness.

I veri Top 10 Broker della Rete:


,Node,Degree,Betweenness_Centrality,PageRank
15,Jeffrey Epstein,18350,0.450158,0.054407
113,unknown person A,6258,0.257399,0.024065
188,Donald Trump,3516,0.088400,0.011851
523,Bill Clinton,910,0.059888,0.002699
1139,Barack Obama,604,0.048315,0.002463
519,Ehud Barak,1365,0.037000,0.004837
117,Alan M. Dershowitz,1527,0.035284,0.005270
810,Edward Snowden,1424,0.032464,0.005429
359,Iran,306,0.030039,0.001185
2383,Steve Bannon,912,0.026092,0.002985


In [5]:
# 5. Esportazione per la fase successiva
output_dir = '../data'
os.makedirs(output_dir, exist_ok=True)

# Salviamo i CSV con le metriche calcolate
df_metrics.to_csv(os.path.join(output_dir, 'nodes_base_metrics.csv'), index=False)
edge_weights.to_csv(os.path.join(output_dir, 'edges_base.csv'), index=False)

# Iniettiamo le metriche dentro il grafo per il salvataggio strutturato
nx.set_node_attributes(G, dict(zip(df_metrics['Node'], df_metrics['Degree'])), 'degree')
nx.set_node_attributes(G, dict(zip(df_metrics['Node'], df_metrics['Betweenness_Centrality'])), 'betweenness')

# Salvataggio in formato standard per grafi (GraphML)
nx.write_graphml(G, os.path.join(output_dir, 'network_investigativa.graphml'))